In [1]:
# ==========================================
# CBOW and Skip-Gram Word Embedding Model
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter

# -------------------------------
# Step 1: Corpus
# -------------------------------
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "i love natural language processing",
    "word embeddings are useful in nlp applications",
    "skip gram and continuous bag of words are popular models",
    "learning nlp is fun and educational"
]

# -------------------------------
# Step 2: Preprocessing
# -------------------------------
words = []
for sentence in corpus:
    words.extend(sentence.split())

vocab = list(set(words))
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for word, i in word_to_ix.items()}
vocab_size = len(vocab)

print("Vocabulary:", word_to_ix)

# -------------------------------
# Step 3: Generate Training Data
# -------------------------------
window_size = 2

# CBOW pairs
cbow_data = []
for sentence in corpus:
    tokens = sentence.split()
    for i in range(window_size, len(tokens) - window_size):
        context = [
            tokens[i-2], tokens[i-1],
            tokens[i+1], tokens[i+2]
        ]
        target = tokens[i]
        cbow_data.append((context, target))

print("\nCBOW training pairs:", cbow_data[:5])

# Skip-gram pairs
skipgram_data = []
for sentence in corpus:
    tokens = sentence.split()
    for i in range(len(tokens)):
        target = tokens[i]
        context_words = tokens[max(0, i-2): i] + tokens[i+1:i+3]
        for context in context_words:
            skipgram_data.append((target, context))

print("\nSkip-gram training pairs:", skipgram_data[:5])

# -------------------------------
# Step 4: CBOW Model
# -------------------------------
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        mean = embeds.mean(dim=0)
        out = self.linear(mean)
        return out

# -------------------------------
# Step 5: Skip-Gram Model
# -------------------------------
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGram, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        out = self.linear(embeds)
        return out

# -------------------------------
# Step 6: Training CBOW
# -------------------------------
embedding_dim = 10
epochs = 50

cbow_model = CBOW(vocab_size, embedding_dim)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(cbow_model.parameters(), lr=0.01)

print("\nTraining CBOW model...")
for epoch in range(epochs):
    total_loss = 0
    for context, target in cbow_data:
        context_idx = torch.tensor([word_to_ix[w] for w in context])
        target_idx = torch.tensor([word_to_ix[target]])

        output = cbow_model(context_idx)
        loss = loss_fn(output.unsqueeze(0), target_idx)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f"CBOW Epoch {epoch+1}, Loss: {total_loss:.4f}")

# -------------------------------
# Step 7: Training Skip-Gram
# -------------------------------
skip_model = SkipGram(vocab_size, embedding_dim)
optimizer = optim.SGD(skip_model.parameters(), lr=0.01)

print("\nTraining Skip-gram model...")
for epoch in range(epochs):
    total_loss = 0
    for target, context in skipgram_data:
        target_idx = torch.tensor([word_to_ix[target]])
        context_idx = torch.tensor([word_to_ix[context]])

        output = skip_model(target_idx)
        loss = loss_fn(output, context_idx)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f"Skip-gram Epoch {epoch+1}, Loss: {total_loss:.4f}")

# -------------------------------
# Step 8: Extract Embeddings
# -------------------------------
word = "learning"

cbow_embedding = cbow_model.embeddings.weight[word_to_ix[word]]
skip_embedding = skip_model.embeddings.weight[word_to_ix[word]]

print("\nEmbedding for 'learning' (CBOW):", cbow_embedding.detach().numpy())
print("Embedding for 'learning' (Skip-gram):", skip_embedding.detach().numpy())

Vocabulary: {'natural': 0, 'learning': 1, 'of': 2, 'is': 3, 'over': 4, 'skip': 5, 'the': 6, 'gram': 7, 'word': 8, 'useful': 9, 'popular': 10, 'processing': 11, 'dog': 12, 'quick': 13, 'applications': 14, 'fun': 15, 'fox': 16, 'are': 17, 'lazy': 18, 'words': 19, 'educational': 20, 'in': 21, 'embeddings': 22, 'jumps': 23, 'i': 24, 'love': 25, 'models': 26, 'and': 27, 'brown': 28, 'continuous': 29, 'nlp': 30, 'bag': 31, 'language': 32}

CBOW training pairs: [(['the', 'quick', 'fox', 'jumps'], 'brown'), (['quick', 'brown', 'jumps', 'over'], 'fox'), (['brown', 'fox', 'over', 'the'], 'jumps'), (['fox', 'jumps', 'the', 'lazy'], 'over'), (['jumps', 'over', 'lazy', 'dog'], 'the')]

Skip-gram training pairs: [('the', 'quick'), ('the', 'brown'), ('quick', 'the'), ('quick', 'brown'), ('quick', 'fox')]

Training CBOW model...
CBOW Epoch 10, Loss: 58.7699
CBOW Epoch 20, Loss: 53.9591
CBOW Epoch 30, Loss: 49.5025
CBOW Epoch 40, Loss: 45.3884
CBOW Epoch 50, Loss: 41.6208

Training Skip-gram model...
S